In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Ingesta - Capa Bronze (CoinGecko)
# MAGIC **Objetivo:** Extraer datos del mercado de criptomonedas desde la API pública de CoinGecko y guardarlos en una tabla Delta cruda.

# COMMAND ----------

import requests
import json
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit

# --- 1. Configuración ---
# URL base de la API pública (sin API key)
BASE_URL = "https://api.coingecko.com/api/v3"
# Endpoint para obtener datos de mercado de las principales monedas
ENDPOINT = f"{BASE_URL}/coins/markets"
# Parámetros: moneda de referencia USD, top 50 por capitalización, 1 página
PARAMS = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 50,
    "page": 1,
    "sparkline": "false"
}
# Nombre de la tabla Delta de salida
TABLE_NAME = "bronze_coingecko_market"

# COMMAND ----------

# --- 2. Función de extracción con manejo de rate limit ---
def fetch_coingecko_data(url, params, max_retries=3):
    """Realiza una petición a CoinGecko con retroceso exponencial en caso de error 429."""
    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=30)
            if response.status_code == 429:
                wait_time = 60 * (attempt + 1)  # Espera 60s, 120s, 180s
                print(f"⚠️ Rate limit alcanzado. Reintentando en {wait_time} segundos...")
                time.sleep(wait_time)
                continue
            response.raise_for_status()
            print("✅ Datos obtenidos exitosamente de CoinGecko.")
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"❌ Error en la petición (intento {attempt + 1}/{max_retries}): {e}")
            if attempt == max_retries - 1:
                raise
            time.sleep(10)
    return None

# COMMAND ----------

# --- 3. Ejecutar la extracción y convertir a DataFrame (Solución al error de tipos) ---

raw_data = fetch_coingecko_data(ENDPOINT, PARAMS)

if raw_data:
    print(f"Se obtuvieron datos de {len(raw_data)} criptomonedas.")
    
    # SOLUCIÓN: Convertimos todos los valores a string para evitar la inferencia de tipos mixtos
    datos_para_spark = []
    for item in raw_data:
        item_str = {str(k): str(v) for k, v in item.items()}
        datos_para_spark.append(item_str)
    
    # Ahora creamos el DataFrame a partir de la lista de diccionarios con valores string
    df_raw = spark.createDataFrame(datos_para_spark)
    
    # Añadir columna de timestamp de ingesta
    df_bronze = df_raw.withColumn("ingestion_timestamp", current_timestamp())
    
    # Escribir como tabla Delta (modo overwrite para la primera carga)
    df_bronze.write.format("delta").mode("overwrite").saveAsTable(TABLE_NAME)
    
    print(f"✅ Tabla Delta '{TABLE_NAME}' creada/actualizada exitosamente.")
    
    # Verificación rápida
    print(f"\nVerificación: número de filas en '{TABLE_NAME}'")
    spark.sql(f"SELECT COUNT(*) as total_filas FROM {TABLE_NAME}").show()
    
    # Mostrar el esquema inferido (ahora todo será StringType)
    print("\nEsquema inferido (todo string):")
    df_bronze.printSchema()
    
    # Mostrar una muestra (los valores se verán como texto)
    print("\nMuestra de los primeros 3 registros:")
    df_bronze.select("id", "symbol", "name", "current_price", "market_cap", "ingestion_timestamp").show(3, truncate=False)
else:
    print("⚠️ No se pudieron obtener datos. Revisá los errores anteriores.")

In [0]:
spark.sql("SHOW TABLES").show()